# Лабораторная работа №19: Работа с длинным контекстом (Long Context Windows)

**ФИО студента:** _______________________
**Группа:** _______________________

## Цель работы
Изучить методы обработки документов, превышающих ограничение на длину контекста языковой модели. Освоить техники разделения текста, рекурсивной суммаризации, иерархической обработки и скользящего окна.

## Теоретические сведения
### Проблема длинного контекста
Большинство LLM имеют ограничение на длину входного контекста (4K-128K токенов). При работе с большими документами (книги, длинные отчёты, транскрипты) необходимо применять специальные стратегии.

### Основные подходы
1. **Chunking (Разделение):** Разбиение текста на части с перекрытием
2. **Map-Reduce:** Параллельная обработка частей и агрегация результатов
3. **Refine:** Последовательное уточнение ответа по частям
4. **Hierarchical Summarization:** Иерархическая суммаризация (дерево)
5. **Sliding Window:** Скользящее окно с памятью контекста

### Метрики качества
- Полнота сохранения информации
- Согласованность ответов
- Время обработки
- Стоимость API вызовов

## Задание
### Уровень 1 (Базовый)
1. Загрузите длинный текст (>10000 символов)
2. Реализуйте простое разбиение на чанки фиксированного размера
3. Примените Map-Reduce для суммаризации каждой части

### Уровень 2 (Продвинутый)
1. Реализуйте рекурсивную суммаризацию с построением дерева
2. Сравните качество суммаризации разных методов
3. Добавьте сохранение промежуточных результатов

### Уровень 3 (Индивидуальный)
1. Реализуйте адаптивное разбиение с учётом семантических границ
2. Создайте систему поиска по длинному документу с контекстом
3. Оптимизируйте количество вызовов LLM для снижения стоимости

## Ход работы

### Шаг 1: Установка зависимостей

In [ ]:
%pip install langchain langchain-openai tiktoken -q

### Шаг 2: Загрузка длинного текста

In [ ]:
# Генерируем длинный текст для примера (симуляция большой книги)
chapters = []
for i in range(10):
    chapter = f"""
Глава {i+1}
В этой главе рассказывается о важных событиях периода {i*10}-{(i+1)*10} годов.
Основные действующие лица: Иван Петров, Мария Сидорова, Алексей Волков.
События развивались в Москве, Санкт-Петербурге и Казани.
Ключевые моменты:
- Развитие технологий искусственного интеллекта
- Изменения в законодательстве о данных
- Международное сотрудничество в области науки
- Экономические реформы и их последствия
- Культурные события и фестивали
Эта глава завершается важными выводами о тенденциях будущего.
Подробный анализ показывает, что...
""" * 50  # Умножаем для длины
    chapters.append(chapter)

long_text = "\n\n".join(chapters)
print(f"Длина текста: {len(long_text)} символов")
print(f"Пример начала:\n{long_text[:500]}...")

### Шаг 3: Подсчёт токенов

In [ ]:
import tiktoken

def count_tokens(text, model="gpt-3.5-turbo"):
    """Подсчитывает количество токенов в тексте"""
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(text))

total_tokens = count_tokens(long_text)
print(f"Общее количество токенов: {total_tokens}")
print(f"Ограничение контекста GPT-3.5: ~16K токенов")
print(f"Необходимо разбиение: {total_tokens > 16000}")

### Шаг 4: Разбиение на чанки

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Настройка сплиттера
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,      # Размер чанка в токенах
    chunk_overlap=200,    # Перекрытие для сохранения контекста
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_text(long_text)
print(f"Количество чанков: {len(chunks)}")
print(f"Средний размер чанка: {sum(len(c) for c in chunks) / len(chunks):.0f} символов")
print(f"\nПервый чанк:\n{chunks[0][:300]}...")

### Шаг 5: Map-Reduce суммаризация

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

# os.environ["OPENAI_API_KEY"] = "your-key-here"
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Промпт для суммаризации одного чанка
map_prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты эксперт по суммаризации. Создай краткую выжимку из следующего текста."),
    ("human", "{text}")
])

map_chain = map_prompt | llm

# Обработка каждого чанка (MAP)
summaries = []
for i, chunk in enumerate(chunks[:5]):  # Ограничим для демо
    print(f"Обработка чанка {i+1}/{len(chunks[:5])}...")
    # response = map_chain.invoke({"text": chunk})
    # summary = response.content
    summary = f"Краткое содержание главы {i+1}: основные события, персонажи, выводы."  # Демо
    summaries.append(summary)

print(f"\nПолучено {len(summaries)} промежуточных суммаризаций")

In [ ]:
# REDUCE: объединение промежуточных результатов
reduce_prompt = ChatPromptTemplate.from_messages([
    ("system", "Объедини следующие краткие содержания в одну связную суммаризацию всего документа."),
    ("human", "{summaries}")
])

reduce_chain = reduce_prompt | llm

combined_summaries = "\n\n".join(summaries)
# final_summary = reduce_chain.invoke({"summaries": combined_summaries})
# print(final_summary.content)

print("=== ФИНАЛЬНАЯ СУММАРИЗАЦИЯ ===")
print("Документ содержит 10 глав, описывающих развитие событий за период...")
print("Ключевые темы: ИИ, законодательство, международное сотрудничество...")
print("Основные выводы касаются будущих тенденций...")

### Шаг 6: Рекурсивная суммаризация (дерево)

In [ ]:
def recursive_summarize(texts, level=0):
    """Рекурсивная суммаризация с построением дерева"""
    if len(texts) <= 1:
        return texts[0] if texts else ""
    
    print(f"Уровень {level}: {len(texts)} текстов")
    
    # Группируем по 2-3 текста
    grouped = []
    for i in range(0, len(texts), 3):
        group = "\n\n".join(texts[i:i+3])
        grouped.append(group)
    
    # Суммаризируем каждую группу
    next_level = []
    for group in grouped:
        # response = map_chain.invoke({"text": group})
        # summary = response.content
        summary = f"Суммаризация группы из {min(3, len(group))} частей"  # Демо
        next_level.append(summary)
    
    return recursive_summarize(next_level, level + 1)

# Запуск рекурсивной суммаризации
print("Запуск рекурсивной суммаризации...")
tree_summary = recursive_summarize(chunks[:10])
print(f"\nИтоговая суммаризация дерева: {tree_summary}")

### Шаг 7: Скользящее окно с памятью

In [ ]:
def sliding_window_summarize(chunks, window_size=3, overlap=1):
    """Скользящее окно с сохранением контекста между окнами"""
    results = []
    context = ""
    
    for i in range(0, len(chunks), window_size - overlap):
        window = chunks[i:i+window_size]
        window_text = context + "\n\n" + "\n\n".join(window)
        
        print(f"Окно {i//window_size + 1}: чанки {i} до {min(i+window_size, len(chunks))}")
        
        # Суммаризация окна
        # response = map_chain.invoke({"text": window_text})
        # summary = response.content
        summary = f"Суммаризация окна {i//window_size + 1} с контекстом"  # Демо
        results.append(summary)
        
        # Сохраняем часть контекста для следующего окна
        context = summary[-500:] if len(summary) > 500 else summary
    
    return results

sliding_results = sliding_window_summarize(chunks[:10])
print(f"\nРезультатов скользящего окна: {len(sliding_results)}")

### Шаг 8: Сравнение методов

In [ ]:
import pandas as pd

# Метрики для сравнения
comparison = pd.DataFrame({
    'Метод': ['Map-Reduce', 'Рекурсивный', 'Скользящее окно'],
    'Кол-во вызовов LLM': [len(chunks[:5]) + 1, 5, 4],
    'Сохранение контекста': ['Частичное', 'Полное', 'Высокое'],
    'Параллелизация': ['Да', 'Нет', 'Частично'],
    'Стоимость': ['Средняя', 'Низкая', 'Средняя']
})

print(comparison.to_string(index=False))

## Контрольные вопросы
1. В каких случаях предпочтителен метод Map-Reduce?
2. Какие преимущества у рекурсивной суммаризации?
3. Как выбрать оптимальный размер чанка и перекрытия?
4. Как минимизировать стоимость обработки длинных документов?

## Выводы
В работе изучены основные методы обработки текстов, превышающих ограничение контекста LLM. Реализованы и сравнены подходы Map-Reduce, рекурсивной суммаризации и скользящего окна. Показаны компромиссы между качеством, стоимостью и скоростью обработки.